# Session 5 — EIM: guided implementation

Download the notebook from the toolbar and use [the course Python environment](https://feelpp.github.io/course-rom/course-rom/setup.html). This is a **guided exercise**: the source data are supplied, but the algorithm is yours to implement. Functions intentionally raise `NotImplementedError` until completed; the starter runs because these exercise functions are not called automatically. Instructor solutions stay outside Antora.


Work one question at a time, show the requested check, then continue after discussion. Keep numerical calculations and plotting in separate cells. The reference graphs in the lecture slides illustrate what to investigate; no reference interpolation basis or completed algorithm is supplied in this notebook.


Read [the EIM method](https://feelpp.github.io/course-rom/rom/hyper-reduction/index.html#eim-algorithm). Today use the construction and interpolation questions; the longer stability and estimator analysis of the historical homework is optional reading.


The Python variable `n` represents the source-grid dimension $N_q$ in the formulas. See [the common notation](https://feelpp.github.io/course-rom/rom/notation.html).

## Notation from the original EIM slides

We use the notation of [the original non-affine slides](https://feelpp.github.io/course-rom/course-rom/_attachments/lecture-rbobm-beamer-non-affine.pdf): the source is $g(x;\mu)$, the basis functions are $\rho_j$, and the coefficients are $\gamma_j(\mu)$.
The interpolant is $I_m^x g=\sum_{j=1}^m\gamma_j(\mu)\rho_j$ in $X_m=\operatorname{span}\{\rho_1,\ldots,\rho_m\}$.
The interpolation points $T_m=\{t^1,\ldots,t^m\}$ are the **magic points**; here these are spatial positions, not times.
The interpolation matrix is $(\mathbb B_m)_{ij}=\rho_j(t^i)$, and its right-hand side has entries $g(t^i;\mu)$.

| Course notation | Python representation |
| --------------- | --------------------- |
| $g(\cdot;\mu_j)$ | `S[:, j]`, computed by `source(x, *params[j])` |
| Basis functions $\rho_j$ | Columns of `rho` |
| Magic points $t^i$ | `x[points]`; `points` stores grid indices |
| $\mathbb B_m$ | `B_eim = rho[points, :]` |
| $\gamma(\mu)$ | Coefficients returned by `interpolation_coefficients` |

In the slides, $S_m=\{\mu^1_{\mathrm{EIM}},\ldots,\mu^m_{\mathrm{EIM}}\}$ is the selected **parameter set**, not the Python snapshot matrix `S`.
A selected snapshot is the **generating function** $\xi_{m+1}$; its interpolation residual is $r_{m+1}=\xi_{m+1}-I_m^x\xi_{m+1}$.
## E1. Observe and collect source snapshots (5 minutes)

The source $g(x;c,w)=0.1+\exp(-((x-c)/w)^2)$ depends non-affinely on location and width. Each column of $S$ is one field on the same 120-point grid. The 27 training parameters and three test parameters are declared separately.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n = 120
x = np.linspace(0., 1., n)
def source(points, center, width):
    return .1+np.exp(-((np.asarray(points)-center)/width)**2)
params = [(c, w) for c in np.linspace(.2, .8, 9) for w in (.08, .14, .2)]
tests = [(.275, .11), (.425, .17), (.675, .13)]
S = np.column_stack([source(x, *p) for p in params])
illustration_parameters = [(.2, .08), (.5, .08), (.8, .08), (.5, .2)]
illustration_fields = np.column_stack([source(x, *p) for p in illustration_parameters])
print(f'EIM starter: {n} grid points, {S.shape[1]} training fields.')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for p, values, style in zip(illustration_parameters, illustration_fields.T, ['-', '--', ':', '-.']):
    ax.plot(x, values, style, label=f'c={p[0]:g}, w={p[1]:g}')
ax.set(xlabel='x', ylabel='Source', title='Fields to approximate; no EIM solution yet')
ax.legend(fontsize=8); ax.grid(alpha=.25)
fig.tight_layout(); plt.show()


Explain why evaluating selected coordinates can be inexpensive. Which array dimension would remain large if you evaluated the whole source before selecting entries?
## E2. Interpolate with a given basis (5–10 minutes)

First suppose basis columns and point indices are known. Write the small interpolation equations on paper. Then complete this function. It should work with one sampled field or a matrix of sampled fields.


In [ ]:
def interpolation_coefficients(basis, indices, sampled_values):
    # basis stores the rho_j columns; coefficients are gamma.
    # Shape check: basis is (n, m), indices has m entries.
    # TODO: extract the square interpolation matrix from the selected rows.
    # TODO: solve the small system; do not explicitly invert a matrix.
    raise NotImplementedError('E2: recover interpolation coefficients')


**Check before continuing:** use a two-column example from the slides. Verify that the reconstructed field agrees with the supplied samples. Ask the instructor to inspect this first building block.
## E3. Construct one new EIM direction (10 minutes)

Given the current interpolation residual matrix, identify the generating function $\xi_{m+1}$ and a magic point where that field has largest residual magnitude. Use the **signed** residual value when normalizing.


In [ ]:
def new_eim_direction(residuals):
    # TODO: identify the joint maximum of abs(residuals), keeping row and column indices.
    # TODO: check that the selected magnitude is safely above zero.
    # TODO: normalize the chosen column by its signed value at the selected row.
    # Return rho_{m+1}, the magic-point index and the generating-snapshot index.
    # Return the new spatial direction and the selected row and column.
    raise NotImplementedError('E3: select and normalize one residual direction')


**Check before continuing:** the new direction equals one at its selected coordinate. If previous coordinates were already interpolated, why must it vanish there? Trace two iterations by hand before writing the loop.
## E4. Assemble the greedy EIM loop (15 minutes)

Reuse E2 and E3. Start with an empty basis; do not attempt a zero-size interpolation solve. Keep previous functions and coordinates. After each enrichment, recompute the residual before recording the final stopping value.


In [ ]:
def eim(snapshots, rank, tolerance=1e-12):
    rho = np.empty((snapshots.shape[0], 0))
    points = []
    history = []
    # TODO: form the current interpolation residual (special case: empty rho).
    # TODO: test its maximum, then check the rank budget.
    # TODO: call E3, append the direction and its point, then repeat.
    # TODO: record the final residual and distinguish tolerance from rank exit.
    raise NotImplementedError('E4: implement the greedy loop')

# After completing E2–E4, uncomment and inspect:
# rho, points, history = eim(S, rank=8)
# B_eim = rho[points, :]
# TODO: verify distinct points, unit diagonal and negligible upper triangle.


## E5. Online interpolation and validation (15 minutes)

For a new pair $(c,w)$, evaluate the source **only** at selected coordinates, then solve the small interpolation equations. Full reconstruction is allowed here for an error plot; it would be separate from a reduced scalar-output query.


In [ ]:
def eim_online(basis, indices, parameter):
    # TODO: evaluate source only at x[indices].
    # TODO: call E2 and reconstruct the field for diagnostics.
    raise NotImplementedError('E5: sampled online interpolation')

# Numerics after your functions work:
# TODO: store exact and approximate test fields and infinity errors for ranks 4, 8, 12.
# TODO: check pointwise interpolation and record the final training residual.


In [ ]:
# Visualization only, using the stored arrays:
# TODO: plot exact/interpolated fields, selected points, and test error versus rank.
# Do not call the source generator or interpolation solver in this plotting cell.


## Checkpoint and DEIM continuation (5 minutes)

Bring the interpolation-matrix checks, selected coordinates and one held-out error plot. Explain why a small training residual is not a continuous-domain guarantee. Only after these work, continue in [the DEIM extension](https://feelpp.github.io/course-rom/course-rom/labs/session06-deim.html), reusing the source data and your interpolation solver. The Lebesgue constant and one-point estimator from the historical homework are optional, not required today.
